In [0]:
from pyspark.sql.functions import col, lit

# Q1 employees
q1_data = [
    ("Ravi",   "Engineering", "Pune",   55000),
    ("Priya",  "HR",          "Mumbai", 42000),
    ("Arjun",  "Engineering", "Delhi",  72000),
    ("Sneha",  "Finance",     "Pune",   61000),
]

# Q2 employees — some same, some new
q2_data = [
    ("Ravi",   "Engineering", "Pune",   60000),  # salary changed
    ("Priya",  "HR",          "Mumbai", 42000),  # same
    ("Rohit",  "Engineering", "Mumbai", 80000),  # new
    ("Meera",  "HR",          "Bangalore",39000), # new
]

cols = ["name", "dept", "city", "salary"]
q1_df = spark.createDataFrame(q1_data, cols)
q2_df = spark.createDataFrame(q2_data, cols)

display(q1_df)
display(q2_df)

name,dept,city,salary
Ravi,Engineering,Pune,55000
Priya,HR,Mumbai,42000
Arjun,Engineering,Delhi,72000
Sneha,Finance,Pune,61000


name,dept,city,salary
Ravi,Engineering,Pune,60000
Priya,HR,Mumbai,42000
Rohit,Engineering,Mumbai,80000
Meera,HR,Bangalore,39000


In [0]:
# Combine both quarters — keeps ALL rows including duplicates
combined = q1_df.union(q2_df)
display(combined)
print("Q1 rows:", q1_df.count())
print("Q2 rows:", q2_df.count())
print("Combined rows:", combined.count())  # 8 rows total

# Add quarter label before union
q1_labeled = q1_df.withColumn("quarter", lit("Q1"))
q2_labeled = q2_df.withColumn("quarter", lit("Q2"))

combined_labeled = q1_labeled.union(q2_labeled)
display(combined_labeled)

name,dept,city,salary
Ravi,Engineering,Pune,55000
Priya,HR,Mumbai,42000
Arjun,Engineering,Delhi,72000
Sneha,Finance,Pune,61000
Ravi,Engineering,Pune,60000
Priya,HR,Mumbai,42000
Rohit,Engineering,Mumbai,80000
Meera,HR,Bangalore,39000


Q1 rows: 4
Q2 rows: 4
Combined rows: 8


name,dept,city,salary,quarter
Ravi,Engineering,Pune,55000,Q1
Priya,HR,Mumbai,42000,Q1
Arjun,Engineering,Delhi,72000,Q1
Sneha,Finance,Pune,61000,Q1
Ravi,Engineering,Pune,60000,Q2
Priya,HR,Mumbai,42000,Q2
Rohit,Engineering,Mumbai,80000,Q2
Meera,HR,Bangalore,39000,Q2


In [0]:
# Remove duplicate rows after union
combined_distinct = q1_df.union(q2_df).distinct()
display(combined_distinct)
# Priya appears in both Q1 and Q2 with same salary
# distinct() removes the duplicate Priya row
print("Distinct rows:", combined_distinct.count())

name,dept,city,salary
Ravi,Engineering,Pune,55000
Priya,HR,Mumbai,42000
Arjun,Engineering,Delhi,72000
Sneha,Finance,Pune,61000
Ravi,Engineering,Pune,60000
Rohit,Engineering,Mumbai,80000
Meera,HR,Bangalore,39000


Distinct rows: 7


In [0]:
# What if column order is different?
q2_reordered = q2_df.select("salary", "name", "city", "dept")

# union() — WRONG result (matches by position)
wrong = q1_df.union(q2_reordered)
display(wrong)  # salary in name column! ❌

# unionByName() — correct (matches by column name)
correct = q1_df.unionByName(q2_reordered)
display(correct)  # ✅ columns matched correctly

# Real rule: always use unionByName in production

---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-5571998775032846>, line 6
      4 # union() — WRONG result (matches by position)
      5 wrong = q1_df.union(q2_reordered)
----> 6 display(wrong)  # salary in name column! ❌
      8 # unionByName() — correct (matches by column name)
      9 correct = q1_df.unionByName(q2_reordered)

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:100, in Display.display_connect_table(self, df, **kwargs)
     97     self.cf_helper.display_streaming_dataframe(df, config, self.streaming_listen

In [0]:
# Find employees who appear in BOTH quarters
# All columns must match exactly

common = q1_df.intersect(q2_df)
display(common)
# Only Priya appears (same name, dept, city, salary)
# Ravi excluded — salary changed (55000 vs 60000)

# intersect with just name column
q1_names = q1_df.select("name")
q2_names = q2_df.select("name")

common_names = q1_names.intersect(q2_names)
display(common_names)
# Ravi and Priya both appear here
# (just name matched, not full row)

name,dept,city,salary
Priya,HR,Mumbai,42000


name
Ravi
Priya


In [0]:
# Find employees in Q1 but NOT in Q2
# Used for: finding who left, data reconciliation

only_q1 = q1_df.exceptAll(q2_df)
display(only_q1)
# Returns: Arjun, Sneha (not in Q2)
# Ravi returned too — his salary changed so full row differs

# except() removes duplicates first then compares
# exceptAll() keeps duplicates in comparison

# Find employees in Q2 but NOT in Q1 (new joiners)
only_q2 = q2_df.exceptAll(q1_df)
display(only_q2)
# Returns: Rohit, Meera (new in Q2)
# Ravi returned — new salary makes him different

name,dept,city,salary
Ravi,Engineering,Pune,55000
Arjun,Engineering,Delhi,72000
Sneha,Finance,Pune,61000


name,dept,city,salary
Ravi,Engineering,Pune,60000
Rohit,Engineering,Mumbai,80000
Meera,HR,Bangalore,39000


In [0]:
# Real world scenario:
# Source system has records — target (Delta table) has records
# Find what's missing in target

source_data = [
    (1, "Ravi",   55000),
    (2, "Priya",  42000),
    (3, "Arjun",  72000),
    (4, "Sneha",  61000),
    (5, "Rohit",  80000),
]

target_data = [
    (1, "Ravi",   55000),
    (2, "Priya",  42000),
    (4, "Sneha",  61000),
]

id_cols = ["id","name","salary"]
source_df = spark.createDataFrame(source_data, id_cols)
target_df = spark.createDataFrame(target_data, id_cols)

# Records in source but missing from target
missing = source_df.exceptAll(target_df)
display(missing)
# Returns: Arjun(3) and Rohit(5) — missing from target ✅

# This is data quality validation — very common in pipelines
print(f"Missing {missing.count()} records from target!")

id,name,salary
3,Arjun,72000
5,Rohit,80000


Missing 2 records from target!


In [0]:
# Real scenario: sales data from 4 regions
# each region sends separate DataFrame
# combine into one master table

north_data = [("North","Jan",50000),("North","Feb",62000)]
south_data = [("South","Jan",45000),("South","Feb",48000)]
east_data  = [("East", "Jan",38000),("East", "Feb",41000)]
west_data  = [("West", "Jan",55000),("West", "Feb",59000)]

scols = ["region","month","sales"]
north = spark.createDataFrame(north_data, scols)
south = spark.createDataFrame(south_data, scols)
east  = spark.createDataFrame(east_data,  scols)
west  = spark.createDataFrame(west_data,  scols)

# Chain multiple unions
all_sales = north \
    .unionByName(south) \
    .unionByName(east) \
    .unionByName(west)

display(all_sales.orderBy("region","month"))
print("Total rows:", all_sales.count())

# Aggregate combined data
from pyspark.sql.functions import sum, avg, round

summary = all_sales.groupBy("month").agg(
    sum("sales").alias("total_sales"),
    round(avg("sales"),0).alias("avg_sales")
).orderBy("month")

display(summary)

# Save
spark.sql("DROP TABLE IF EXISTS all_regions_sales")
all_sales.write.format("delta").mode("overwrite") \
    .saveAsTable("all_regions_sales")
print("✅ Saved!")

region,month,sales
East,Feb,41000
East,Jan,38000
North,Feb,62000
North,Jan,50000
South,Feb,48000
South,Jan,45000
West,Feb,59000
West,Jan,55000


Total rows: 8


month,total_sales,avg_sales
Feb,210000,52500.0
Jan,188000,47000.0


✅ Saved!
